# Load test — Zarr NWB with AIND metadata

Only dependencies: `hdmf_zarr` (brings `NWBZarrIO`) + stdlib `json` + `numpy`. No custom class import, no extension registration.

In [2]:
from hdmf_zarr import NWBZarrIO
import json
import numpy as np

write_save_file = '/root/capsule/data/test_nwb/test.nwb.zarr'

In [3]:
# One IO, kept open for the rest of the notebook so lazy datasets stay accessible.
io = NWBZarrIO(write_save_file, 'r')
nwb = io.read()

# AIND metadata lives at /general/aind_metadata/json_data (LabMetaData containers
# sit directly under /general/, not under /general/lab_meta_data/).
raw = io.file['general/aind_metadata/json_data'][()]
if isinstance(raw, np.ndarray):
    raw = raw.item()
if isinstance(raw, bytes):
    raw = raw.decode()
meta = json.loads(raw)

print('metadata files:', list(meta.keys()))

metadata files: ['acquisition', 'data_description', 'instrument', 'metadata.nd', 'procedures', 'subject']


In [4]:
print('acquisition subject_id:', meta['acquisition'].get('subject_id'))
print('subject genotype:', meta['subject'].get('subject_details', {}).get('genotype'))

acquisition subject_id: ZS062
subject genotype: Dbh-Cre-KI/wt


## Trials

In [5]:
trials_df = nwb.trials.to_dataframe()
print(f'{len(trials_df)} rows, {len(trials_df.columns)} columns')
print('ragged sample right_reward_times[0:5]:', trials_df['right_reward_times'].head(5).tolist())
trials_df.head()

564 rows, 75 columns
ragged sample right_reward_times[0:5]: [array([], dtype=float64), array([], dtype=float64), array([], dtype=float64), array([], dtype=float64), array([], dtype=float64)]


,start_time,stop_time,animal_response,rewarded_historyL,rewarded_historyR,delay_start_time,goCue_start_time,reward_outcome_time,bait_left,bait_right,...,lickspout_position_y2,reward_size_left,reward_size_right,lick_lat,trial_ind,right_reward_times,left_reward_times,choice_time_trial,auto_manual_trial,extra_reward
id,,,,,,,,,,,,,,,,,,,,,
0,7.165777e+06,7.165786e+06,1.0,False,False,7.165783e+06,7.165784e+06,7.165785e+06,True,False,...,10.55,2.0,2.0,0.300768,0,[],[],0.300768,False,False
1,7.165786e+06,7.165792e+06,1.0,False,False,7.165790e+06,7.165791e+06,7.165791e+06,True,False,...,10.55,2.0,2.0,0.342752,1,[],[],0.342752,False,False
2,7.165792e+06,7.165797e+06,0.0,True,False,7.165794e+06,7.165795e+06,7.165796e+06,True,False,...,10.55,2.0,2.0,0.203712,2,[],[0.4252159995958209],0.203712,False,False
3,7.165797e+06,7.165806e+06,1.0,False,False,7.165800e+06,7.165805e+06,7.165805e+06,True,False,...,10.55,2.0,2.0,0.342688,3,[],[],0.342688,False,False
4,7.165806e+06,7.165811e+06,1.0,False,False,7.165808e+06,7.165809e+06,7.165810e+06,True,False,...,10.55,2.0,2.0,0.274656,4,[],[],0.274656,False,False


## Units

In [6]:
units_df = nwb.units.to_dataframe()
print(f'{len(units_df)} rows, {len(units_df.columns)} columns')
print('spike counts per unit (first 5):', [len(units_df['spike_times'].iloc[i]) for i in range(5)])
units_df.head()

269 rows, 90 columns
spike counts per unit (first 5): [1751, 1817, 2298, 2256, 105437]


,unit_id,maximum_increase_of_p(response|laser)_from_baseline,maximum_increase_of_p(response|laser),mean_p(response|laser),spike_latency_at_maximum_p(response|laser),mean_spike_latency,waveform_euclidean_distance_at_maximum_p(response|laser),waveform_correlation_at_maximum_p(response|laser),opto_tagged_lowbar,amplitude_of_spike_waveform,...,half_width,shank,drift_std,amplitude_cv_range,estimated_z,drift_ptp,nn_hit_rate,peak_to_valley,amplitude_cv_median,spike_times
id,,,,,,,,,,,,,,,,,,,,,
0,0,0.000018,-0.000018,-0.000018,NaN,NaN,NaN,NaN,False,813.711151,...,0.00047,,NaN,NaN,1.87,NaN,0.132000,0.00084,NaN,"[7165387.737186914, 7165486.199493588, 7165660..."
1,1,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,False,481.058090,...,0.00022,,NaN,NaN,1.02,NaN,0.097333,0.00021666666666666666,NaN,"[7165338.616698973, 7165387.736820251, 7165387..."
2,2,0.000326,0.249674,0.069356,0.001172,0.001172,2.091186,0.280532,False,413.935349,...,0.00021666666666666666,,NaN,NaN,1.00,NaN,0.892000,0.0012466666666666668,NaN,"[7165366.707633152, 7165382.523060174, 7165382..."
3,3,0.003919,0.996081,0.960714,0.001590,0.001590,1.056457,0.703245,False,294.314743,...,0.0006933333333333333,,NaN,NaN,3.98,NaN,0.204000,0.0010766666666666667,NaN,"[7165338.262462474, 7165338.615932299, 7165382..."
4,4,0.369391,0.230609,0.059547,0.012854,0.012854,0.079480,0.993835,False,131.717407,...,0.00019333333333333333,,5.790708,0.08229,21.58,21.641392,0.972000,0.00040666666666666667,0.16073,"[7165321.501828095, 7165321.625861303, 7165321..."


## Acquisition TimeSeries

In [7]:
for name, ts in nwb.acquisition.items():
    n = len(ts.timestamps)
    print(f'{name}: {n} timestamps, unit={ts.unit}, range=[{ts.timestamps[0]}, {ts.timestamps[-1]}]')

left_lick_time: 1942 timestamps, unit=second, range=[7165785.370752, 7170453.627008]
left_reward_delivery_time: 131 timestamps, unit=second, range=[7165795.771488, 7170046.026496]
right_lick_time: 2741 timestamps, unit=second, range=[7165778.541696, 7171771.0448]
right_reward_delivery_time: 159 timestamps, unit=second, range=[7165905.725504, 7170339.078496]


## Processing modules

In [8]:
for pm_name, pm in nwb.processing.items():
    print(f'{pm_name}:')
    for obj_name, obj in pm.data_interfaces.items():
        print(f'  {obj_name}: {type(obj).__name__}')

In [10]:
nwb.lab_meta_data

{'aind_metadata': aind_metadata abc.AindMetadata at 0x140372540701920
 Fields:
   json_data: {"acquisition": {"acquisition_end_time": "2021-05-01T20:46:52.115000-04:00", "acquisition_start_time": "2021-05-01T19:48:01-04:00", "acquisition_type": "Behavioral foraging", "calibrations": [], "coordinate_system": null, "data_streams": [{"active_devices": ["Neuralynx Ephys Assembly", "Pupil camera", "Arduino", "Lick spout assembly"], "code": null, "configurations": [{"device_name": "Neuralynx Ephys Assembly", "manipulator": {"coordinate_system": {"axes": [{"direction": "Posterior_to_anterior", "name": "AP", "object_type": "Axis"}, {"direction": "Left_to_right", "name": "ML", "object_type": "Axis"}, {"direction": "Superior_to_inferior", "name": "SI", "object_type": "Axis"}], "axis_unit": "millimeter", "name": "BREGMA_ARI", "object_type": "Coordinate system", "origin": "Bregma"}, "device_name": "Drive system", "local_axis_positions": {"object_type": "Translation", "translation": [0, 0, 0]}, "object_type": "Manipulator config"}, "modules": [], "object_type": "Ephys assembly config", "probes": []}], "connections": [], "modalities": [{"abbreviation": "behavior", "name": "Behavior"}, {"abbreviation": "ecephys", "name": "Extracellular electrophysiology"}], "notes": "ephys recording session", "object_type": "Data stream", "stream_end_time": "2021-05-01T20:46:52.115000-04:00", "stream_start_time": "2021-05-01T19:48:01-04:00"}], "describedBy": "https://raw.githubusercontent.com/AllenNeuralDynamics/aind-data-schema/main/src/aind_data_schema/core/acquisition.py", "ethics_review_id": ["MO19M432"], "experimenters": ["zhixiao su"], "instrument_id": "hopkins_295F_nlyx", "maintenance": [], "notes": "ephys recording session for subject ZS062. tetrode depth = 4.429. session_notes = 'Good'", "object_type": "Acquisition", "protocol_id": null, "schema_version": "2.0.34", "specimen_id": null, "stimulus_epochs": [{"active_devices": ["Speaker"], "code": {"container": null, "core_dependency": null, "input_data": null, "language": null, "language_version": null, "name": "dynamic-foraging-task", "object_type": "Code", "parameters": {"BlockBeta": "15", "BlockMax": "35", "BlockMin": "20", "DelayBeta": "0.0", "DelayMax": "1.0", "DelayMin": "1.0", "ITIBeta": "3.0", "ITIMax": "15.0", "ITIMin": "2.0", "LeftValue_volume": "2.50", "RightValue_volume": "2.50", "Task": "Uncoupled Without Baiting", "amplitude_db": 60, "frequency_unit": "hertz", "go_cue_frequency": 7500, "no_go_cue_frequency": 15000, "reward_probability": "0.1, 0.5, 0.9", "sample_frequency": 96000, "stimulus_duration_ms": 500}, "run_script": null, "url": "https://github.com/JeremiahYCohenLab/sueBehavior.git", "version": null}, "configurations": [{"device_name": "Speaker", "object_type": "Speaker config", "volume": 60, "volume_unit": "decibels"}], "curriculum_status": null, "notes": null, "object_type": "Stimulus epoch", "performance_metrics": {"object_type": "Performance metrics", "output_parameters": {}, "reward_consumed_during_epoch": "675.0", "reward_consumed_unit": "microliter", "trials_finished": 424, "trials_rewarded": 270, "trials_total": 424}, "stimulus_end_time": "2021-05-01T20:46:52.115000-04:00", "stimulus_modalities": ["Auditory"], "stimulus_name": "Behavioral foraging task", "stimulus_start_time": "2021-05-01T19:48:01-04:00", "training_protocol_name": null}], "subject_details": {"anaesthesia": null, "animal_weight_post": null, "animal_weight_prior": "24.1", "mouse_platform_name": "mouse_tube_foraging_hopkins", "object_type": "Acquisition subject details", "reward_consumed_total": "0.675", "reward_consumed_unit": "milliliter", "weight_unit": "gram"}, "subject_id": "ZS062"}, "data_description": {"creation_time": "2021-05-01T19:48:01-04:00", "data_level": "derived", "data_summary": "Behavioral ephys recording session for subject ZS062", "describedBy": "https://raw.githubusercontent.com/AllenNeuralDynamics/aind-data-schema/main/src/aind_data_schema/core/data_description.py", "funding_source": [{"f